# P05 - Basic RAG

## RAG 사전단계 (저장)
vectorstore 2개를 생성하여 각각 2개의 PDF를 load-split-embed-store
1. **load -> PyPDF Loader 사용** -> (`uv add pypdf langchain-community`)
1. split -> 적절한 split 기준을 찾아서 적용 (우선은 `RecursiveCharacterTextSplitter` 사용)
1. embed -> OpenAI `text-embedding-3-small` 사용
1. store -> `InMemoryVectorStore` 사용하되, 총 2개의 vectorstore 생성해야함. 변수명은
    - `nvda_vectorstore`
    - `googl_vectorstore`

## Agent 구현단계
**Agent에 2(3)개의 Tool 주기**
1. `nvda_vectorstore` 를 감싸고 있는 Tool
1. `googl_vectorstore` 를 감싸고 있는 Tool
1. (Optional) `TavilySearchTool` 제공 가능
1. System Prompt 와 Tool Description 을 잘 작성하여 필요한 경우 필요한 Tool 호출하도록 제작

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Set Vectorstore

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True  # 쪼개진 chunk 의 시작 index를 기록
)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

C:\Users\sesac_seocho\AppData\Local\Temp\ipykernel_14616\3207699851.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
# Google 
loader = PyPDFLoader('./pdfs/GOOG-10-K-2025.pdf')
docs = loader.load()

chunks = text_splitter.split_documents(docs)
googl_vectorstore = InMemoryVectorStore(embedding=embeddings)

ids = googl_vectorstore.add_documents(documents=chunks)
print(len(ids))

461


In [5]:
# NVDIA Vectorstore
loader = PyPDFLoader('./pdfs/NVDA-10-K-2025.pdf',)
docs = loader.load()

chunks = text_splitter.split_documents(docs)
nvda_vectorstore = InMemoryVectorStore(embedding=embeddings)

ids = nvda_vectorstore.add_documents(documents=chunks)
print(len(ids))

481


In [6]:
# Tool 만들기
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

@tool
def search_google_10k(query: str):
    """Retrieve info from GOOGLE 10-k report 2025 to help answer a query about """
    docs = googl_vectorstore.similarity_search(query, k=5)
    result = '----------------------\n\n'.join(map(lambda doc: doc.page_content, docs))
    return result


@tool
def search_nvda_10k(query: str):
    """Retrieve info from NVIDA 10-k report 2025 to help answer a query about """
    docs = nvda_vectorstore.similarity_search(query, k=5)
    result = '----------------------\n\n'.join(map(lambda doc: doc.page_content, docs))
    return result

tavily_search_tool = TavilySearch(max_results=5, topic="news")
tavily_search_tool.name 


'tavily_search'

In [7]:
from langchain.agents import create_agent

AGNET_SYSTEM_PROMPT = '''# System Prompt: Financial Intelligence & Market Analyst Agent

## Purpose
You are an expert Financial Analyst and Market Intelligence Agent Answer in KOREAN.
Your goal is to provide comprehensive, accurate, and deeply analytical insights regarding NVIDIA and Google (Alphabet). You have direct access to their official 10-K annual reports and real-time market news.

## Available Tools
1. `search_nvda_10k`: Searches the VectorStore containing NVIDIA's official 10-K financial reports. Use this for historical financial data, risk factors, official business segments, and executive commentary.
2. `search_google_10k`: Searches the VectorStore containing Google's official 10-K financial reports. Use this for historical financial data, revenue breakdowns (e.g., Cloud, Search, YouTube), and official corporate strategies.
3. `tavily_search`: Searches the live internet for the latest news, recent stock movements, recent product launches, and current market sentiment.

## Operational Rules & Chain of Thought

### 1. Tool Selection Strategy
*   **Historical & Structural Data:** For questions about past fiscal years, balance sheets, income statements, or official risk factors, ALWAYS prioritize the `10k_search` tools.
*   **Current & Dynamic Data:** For questions about current stock prices, recent events, breaking news, or Q3/Q4 updates that happened after the last 10-K filing, ALWAYS use `tavily_search`.
*   **Comparative Analysis:** If a user asks to compare historical performance with current market trends, you MUST cross-reference by calling BOTH the respective 10-K tool and the Tavily search tool.

### 2. Temporal Precision
*   The current year is **2026**. Keep this in mind when evaluating "recent" news or determining which 10-K report is the most current.
*   Always state the specific fiscal year or date when presenting financial metrics (e.g., "According to NVIDIA's FY2025 10-K..."). Never state past events as future events.

### 3. Accuracy, Grounding & Tone
*   **No Hallucination:** Rely strictly on the facts retrieved from the tools. If the information is not present in the 10-K or verified news, state clearly: "I cannot find verified data for this specific metric in the available sources."
*   **Professional Tone:** Maintain a neutral, professional, and objective financial analyst persona. Avoid overly emotional language regarding stock trends.
*   **Source Citation:** When answering from the 10-K reports, mention that the data comes from the official filing. When answering from Tavily, mention the specific news sources or dates if available.

### 4. Handling Constraints
*   If the user asks "other than X" or "besides X", strictly exclude any concepts, products, or metrics already discussed in the conversation history.

'''

agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[search_google_10k, search_nvda_10k, tavily_search_tool],
    system_prompt=AGNET_SYSTEM_PROMPT,
)

In [8]:
state = {
    'messages': [
        {'role': 'user', 'content': 'NVIDA의 2025년 매출과 영업이익을 작년과 대비하여 분석해줘.'}
    ]
}
for event in agent.stream(state, stream_mode='values'):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

NVIDA의 2025년 매출과 영업이익을 작년과 대비하여 분석해줘.
================================== Ai Message ==================================
Tool Calls:
  search_nvda_10k (call_4Rg9kCu3Sm070QIWb6ifAAne)
 Call ID: call_4Rg9kCu3Sm070QIWb6ifAAne
  Args:
    query: FY2025 revenue operating income FY2024 revenue operating income NVIDIA 10-K official filing net revenue operating income compare year over year
================================= Tool Message =================================
Name: search_nvda_10k

Table of Contents
NVIDIA Corporation and Subsidiaries
Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026 Jan 26, 2025 Jan 28, 2024
Revenue $ 215,938 $ 130,497 $ 60,922 
Cost of revenue 62,475 32,639 16,621 
Gross profit 153,463 97,858 44,301 
Operating expenses
Research and development 18,497 12,914 8,675 
Sales, general and administrative 4,579 3,491 2,654 
Total operating expenses 